In [1]:
# Install the library (quiet)
!pip install -q sentence-transformers scikit-learn pandas numpy

# Imports
import os, json, random, time, platform
import numpy as np
import pandas as pd
import torch
import sklearn
import sentence_transformers

# ONE config cell: change settings only here
CONFIG = {
    "seed": 42,
    "model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "top_k": 3,
    "base_dir": "/content/quickpull_smartreply",
    "data_dir": "/content/quickpull_smartreply/data",
    "deploy_dir": "/content/quickpull_smartreply/deploy",
}

# Set seeds so results repeat
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

# Create folders
for d in [CONFIG["base_dir"], CONFIG["data_dir"], CONFIG["deploy_dir"]]:
    os.makedirs(d, exist_ok=True)

# Print versions
print("Python:", platform.python_version())
print("sentence-transformers:", sentence_transformers.__version__)
print("torch:", torch.__version__)
print("scikit-learn:", sklearn.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("GPU available:", torch.cuda.is_available())
print("Folders ready:", os.listdir(CONFIG["base_dir"]))

Python: 3.13.16
sentence-transformers: 5.7.0
torch: 2.11.0+cpu
scikit-learn: 1.6.1
pandas: 2.2.3
numpy: 2.1.3
GPU available: False
Folders ready: ['data', 'deploy']


In [2]:
# Reply bank: (id, role, intent, text). role = who SENDS the reply
bank_rows = [
# ---- Driver replies ----
("D01","driver","greeting","Hello!"),
("D02","driver","greeting","Hi, I'm your driver."),
("D03","driver","on_my_way","I'm on my way."),
("D04","driver","on_my_way","On my way to you."),
("D05","driver","on_my_way","Heading to your pickup now."),
("D06","driver","few_minutes_away","I'm a few minutes away."),
("D07","driver","few_minutes_away","Almost there, just a little more."),
("D08","driver","arrived_pickup","I have arrived at the pickup."),
("D09","driver","arrived_pickup","I'm here at the pickup point."),
("D10","driver","arrived_pickup","Reached your pickup location."),
("D11","driver","waiting_at_pickup","I'm waiting at the pickup point."),
("D12","driver","waiting_at_pickup","Waiting for you here."),
("D13","driver","stuck_in_traffic","Stuck in traffic, sorry."),
("D14","driver","stuck_in_traffic","Heavy traffic, I'll be there soon."),
("D15","driver","ask_location","Where exactly are you?"),
("D16","driver","ask_location","Can you share your exact location?"),
("D17","driver","ask_location","Please send your exact location."),
("D18","driver","ask_landmark","Any landmark near you?"),
("D19","driver","ask_landmark","Can you tell me a nearby landmark?"),
("D20","driver","check_vehicle_number","Please check the vehicle number in the app."),
("D21","driver","check_vehicle_number","Match the vehicle number in the app."),
("D22","driver","cant_find_you","I can't find you here."),
("D23","driver","cant_find_you","I can't see you at the pickup."),
("D24","driver","please_wait","Please wait a moment."),
("D25","driver","please_wait","Please give me a moment."),
("D26","driver","acknowledge","Ok."),
("D27","driver","acknowledge","Sure."),
("D28","driver","acknowledge","Okay, noted."),
("D29","driver","thanks","Thank you!"),
("D30","driver","thanks","Thanks a lot."),
("D31","driver","thanks","Thanks, appreciate it."),
("D32","driver","sorry_delay","Sorry for the delay."),
("D33","driver","sorry_delay","Sorry to keep you waiting."),
("D34","driver","confirm_dropoff","Is the drop-off point as per the app?"),
("D35","driver","confirm_dropoff","Confirming the drop-off point on the map?"),
("D36","driver","confirm_dropoff","Drop at the app location, right?"),
("D37","driver","cash_payment","Yes, cash is fine."),
("D38","driver","cash_payment","Cash payment is fine."),
("D39","driver","goodbye","Goodbye, take care!"),
("D40","driver","goodbye","Have a nice day!"),
# ---- Rider replies ----
("R01","rider","greeting","Hello!"),
("R02","rider","greeting","Hi, I'm your rider."),
("R03","rider","coming_now","Coming now."),
("R04","rider","coming_now","I'm coming, one moment."),
("R05","rider","coming_now","Walking to you now."),
("R06","rider","please_wait","Please wait a moment."),
("R07","rider","please_wait","Please wait, I'm coming."),
("R08","rider","please_wait","Give me a moment please."),
("R09","rider","at_pickup_point","I'm at the pickup point on the map."),
("R10","rider","at_pickup_point","I'm at the pinned location."),
("R11","rider","at_pickup_point","I'm standing at the pickup point."),
("R12","rider","can_see_you","I can see you."),
("R13","rider","can_see_you","Yes, I see your vehicle."),
("R14","rider","cant_see_you","I can't see you. Where are you?"),
("R15","rider","cant_see_you","Where are you exactly?"),
("R16","rider","cant_see_you","I can't see your vehicle here."),
("R17","rider","acknowledge","Ok."),
("R18","rider","acknowledge","Sure."),
("R19","rider","acknowledge","Okay, got it."),
("R20","rider","thanks","Thank you!"),
("R21","rider","thanks","Thanks a lot."),
("R22","rider","sorry_late","Sorry I'm late."),
("R23","rider","sorry_late","Sorry, I'll be there shortly."),
("R24","rider","confirm_dropoff","Yes, drop-off is as per the app."),
("R25","rider","confirm_dropoff","Yes, the app location is correct."),
("R26","rider","confirm_dropoff","Please drop me at the app location."),
("R27","rider","stop_next_corner","Can we stop at the next corner?"),
("R28","rider","stop_next_corner","Please stop at the next corner."),
("R29","rider","cash_payment","I will pay in cash."),
("R30","rider","cash_payment","I'll pay cash at drop-off."),
("R31","rider","goodbye","Goodbye, thank you!"),
("R32","rider","goodbye","Bye, have a nice day!"),
("R33","rider","ask_how_far","How far are you?"),
("R34","rider","ask_how_far","How long will you take?"),
("R35","rider","come_to_pickup","Please come to the pickup point."),
("R36","rider","come_to_pickup","Please come to the pinned location."),
("R37","rider","ask_ac","Could you turn the AC on?"),
("R38","rider","ask_ac","Please turn on the AC."),
("R39","rider","ask_slow_down","Please drive a bit slower."),
("R40","rider","ask_slow_down","Could you slow down a little?"),
]

bank = pd.DataFrame(bank_rows, columns=["id", "role", "intent", "text"])

# Safety checks: unique ids, max 8 words, no digits
assert bank["id"].is_unique, "Duplicate ids"
assert (bank["text"].str.split().str.len() <= 8).all(), "A reply has more than 8 words"
assert not bank["text"].str.contains(r"\d").any(), "A reply contains a digit"

# Save to CSV
bank_path = os.path.join(CONFIG["data_dir"], "reply_bank.csv")
bank.to_csv(bank_path, index=False)

# Show counts
print("Saved:", bank_path)
print("Total replies:", len(bank))
print("\nReplies per role:\n", bank["role"].value_counts())
print("\nIntents per role:\n", bank.groupby("role")["intent"].nunique())
print("\nReplies per role and intent:\n", bank.groupby(["role", "intent"]).size())

Saved: /content/quickpull_smartreply/data/reply_bank.csv
Total replies: 80

Replies per role:
 role
driver    40
rider     40
Name: count, dtype: int64

Intents per role:
 role
driver    17
rider     17
Name: intent, dtype: int64

Replies per role and intent:
 role    intent              
driver  acknowledge             3
        arrived_pickup          3
        ask_landmark            2
        ask_location            3
        cant_find_you           2
        cash_payment            2
        check_vehicle_number    2
        confirm_dropoff         3
        few_minutes_away        2
        goodbye                 2
        greeting                2
        on_my_way               3
        please_wait             2
        sorry_delay             2
        stuck_in_traffic        2
        thanks                  3
        waiting_at_pickup       2
rider   acknowledge             3
        ask_ac                  2
        ask_how_far             2
        ask_slow_down         

In [3]:
# Dev messages: (id, sender_role, text, acceptable_intents, style)
dev_rows = [
# ---- Rider -> driver ----
("DV01","rider","hey driver, where r u now","on_my_way, few_minutes_away, stuck_in_traffic","informal"),
("DV02","rider","hii","greeting","informal"),
("DV03","rider","I'm standing near the main gate","acknowledge, on_my_way","clean"),
("DV04","rider","are you coming? its been so long","on_my_way, sorry_delay, few_minutes_away","informal"),
("DV05","rider","kitna time lagega bhaiya","few_minutes_away, on_my_way, stuck_in_traffic","hinglish"),
("DV06","rider","Can I pay by cash at the end of the trip?","cash_payment","clean"),
("DV07","rider","thx for the ride","thanks, goodbye","informal"),
("DV08","rider","i cant see ur car anywhere","arrived_pickup, ask_location, waiting_at_pickup","informal"),
("DV09","rider","Please drop me at the metro station gate","confirm_dropoff, acknowledge","clean"),
("DV10","rider","bhaiya main pickup point pe hu","acknowledge, on_my_way, few_minutes_away","hinglish"),
("DV11","rider","sorry, need a moment, coming down","acknowledge, waiting_at_pickup","informal"),
("DV12","rider","which car number are u in","check_vehicle_number","informal"),
("DV13","rider","Good evening","greeting","clean"),
("DV14","rider","ok see you soon","acknowledge, thanks","informal"),
("DV15","rider","Thank you so much, great driving","thanks, goodbye","clean"),
("DV16","rider","driver ji aap kahan ho","on_my_way, few_minutes_away, stuck_in_traffic","hinglish"),
("DV17","rider","app shows u r 5 min away is that right","few_minutes_away, on_my_way","informal"),
("DV18","rider","Can you wait outside the mall entrance?","acknowledge, waiting_at_pickup","clean"),
("DV19","rider","hmm","none","informal"),
("DV20","rider","what is the score of the match","none","clean"),
# ---- Driver -> rider ----
("DV21","driver","Good morning ma'am, driver here","greeting","clean"),
("DV22","driver","I've reached the spot, where are you?","at_pickup_point, coming_now, please_wait","clean"),
("DV23","driver","im outside, pls come fast","coming_now, please_wait, sorry_late","informal"),
("DV24","driver","kaha ho aap","at_pickup_point, coming_now, cant_see_you","hinglish"),
("DV25","driver","Can you see a white sedan near the signal?","can_see_you, cant_see_you","clean"),
("DV26","driver","how many more mins sir","coming_now, please_wait, ask_how_far","informal"),
("DV27","driver","Would you like the AC on?","ask_ac, acknowledge","clean"),
("DV28","driver","is this the corect drop point","confirm_dropoff","informal"),
("DV29","driver","we r reaching in 5 min","acknowledge, thanks","informal"),
("DV30","driver","Payment will be cash or online?","cash_payment","clean"),
("DV31","driver","Thank you for choosing us, stay safe","thanks, goodbye","clean"),
("DV32","driver","sir kaha rukna hai","stop_next_corner, confirm_dropoff","hinglish"),
("DV33","driver","stuck in jam, will be late","acknowledge, ask_how_far, please_wait","informal"),
("DV34","driver","Ride completed, take care","goodbye, thanks","clean"),
("DV35","driver","pls dont be late, wait limit is almost over","sorry_late, coming_now","informal"),
("DV36","driver","aap pickup point pe ho?","at_pickup_point, coming_now","hinglish"),
("DV37","driver","Are you standing at the spot shown on the map?","at_pickup_point, coming_now","clean"),
("DV38","driver","cant locate u, send ur location","at_pickup_point, cant_see_you, please_wait","informal"),
("DV39","driver","lol what","none","informal"),
("DV40","driver","the match was crazy yesterday","none","clean"),
]

dev = pd.DataFrame(dev_rows, columns=["id","sender_role","text","acceptable_intents","style"])

# Store intents separated by "|" (safe for CSV)
dev["acceptable_intents"] = dev["acceptable_intents"].apply(
    lambda s: "none" if s.strip() == "none" else "|".join(x.strip() for x in s.split(",")))
dev["answerable"] = dev["acceptable_intents"] != "none"

# Check: every intent exists in the bank for the role that will REPLY
bank = pd.read_csv(os.path.join(CONFIG["data_dir"], "reply_bank.csv"))
reply_role = {"rider": "driver", "driver": "rider"}
for _, r in dev[dev["answerable"]].iterrows():
    valid = set(bank[bank["role"] == reply_role[r["sender_role"]]]["intent"])
    bad = [i for i in r["acceptable_intents"].split("|") if i not in valid]
    assert not bad, f"{r['id']}: unknown intents {bad}"
assert dev["id"].is_unique

# Save and show counts
dev.to_csv(os.path.join(CONFIG["data_dir"], "dev_messages.csv"), index=False)
print("Saved dev messages:", len(dev))
print("\nPer sender role:\n", dev["sender_role"].value_counts())
print("\nPer style:\n", dev["style"].value_counts())
print("\nAnswerable or not:\n", dev["answerable"].value_counts())

Saved dev messages: 40

Per sender role:
 sender_role
rider     20
driver    20
Name: count, dtype: int64

Per style:
 style
informal    18
clean       16
hinglish     6
Name: count, dtype: int64

Answerable or not:
 answerable
True     36
False     4
Name: count, dtype: int64


In [4]:
# Test messages: (id, sender_role, text, acceptable_intents, style)
test_rows = [
# ---- Rider -> driver ----
("T01","rider","hello sir","greeting","informal"),
("T02","rider","are u close by?","on_my_way, few_minutes_away, stuck_in_traffic","informal"),
("T03","rider","I'm waiting near the petrol pump","acknowledge, on_my_way, few_minutes_away","clean"),
("T04","rider","Please be quick, I have a train to catch","on_my_way, few_minutes_away, sorry_delay","clean"),
("T05","rider","bhaiya kitni door ho","few_minutes_away, on_my_way, stuck_in_traffic","hinglish"),
("T06","rider","I only have cash, is that ok?","cash_payment","clean"),
("T07","rider","thnks, smooth ride","thanks, goodbye","informal"),
("T08","rider","wher is the cab? i dont see it","arrived_pickup, waiting_at_pickup, ask_location","informal"),
("T09","rider","Could you drop me one block before the destination?","confirm_dropoff, acknowledge","clean"),
("T10","rider","main gate ke paas khada hu","acknowledge, on_my_way, few_minutes_away","hinglish"),
("T11","rider","give me 2 mins, locking my door","acknowledge, waiting_at_pickup","informal"),
("T12","rider","whats ur plate number","check_vehicle_number","informal"),
("T13","rider","Good afternoon!","greeting","clean"),
("T14","rider","alrite, will wait here","acknowledge, thanks","informal"),
("T15","rider","Really appreciate the help today","thanks, goodbye","clean"),
("T16","rider","bhai gaadi kahan hai","on_my_way, few_minutes_away, stuck_in_traffic","hinglish"),
("T17","rider","app says arriving in 2 mins but nothing yet","few_minutes_away, on_my_way, sorry_delay","informal"),
("T18","rider","Please wait at the pharmacy, I'll walk over","acknowledge, waiting_at_pickup","clean"),
("T19","rider","my pin is wrong, im 50m ahead","ask_location, ask_landmark, acknowledge","informal"),
("T20","rider","drop me at the bus stand pls","confirm_dropoff, acknowledge","informal"),
("T21","rider","r u still there?","waiting_at_pickup, arrived_pickup, cant_find_you","informal"),
("T22","rider","driver sahab late ho gaya kya","sorry_delay, stuck_in_traffic, on_my_way","hinglish"),
("T23","rider","ok bye, have a good one","goodbye, thanks","informal"),
("T24","rider","The pickup spot is in the lane behind the bank","acknowledge, on_my_way","clean"),
("T25","rider","paisa cash me dunga","cash_payment","hinglish"),
("T26","rider","???","none","informal"),
("T27","rider","what is the weather like","none","clean"),
("T28","rider","asdf","none","informal"),
("T29","rider","Do you like cricket?","none","clean"),
("T30","rider","Which phone do you use?","none","clean"),
# ---- Driver -> rider ----
("T31","driver","Hello sir, I'm your captain","greeting","clean"),
("T32","driver","reached the location, pls come","coming_now, please_wait, sorry_late","informal"),
("T33","driver","wher r u standing?","at_pickup_point, coming_now, cant_see_you","informal"),
("T34","driver","kitni der me aaoge","coming_now, please_wait, sorry_late","hinglish"),
("T35","driver","Do you see my car? Blue hatchback","can_see_you, cant_see_you","clean"),
("T36","driver","need more time? i can wait 3 min","please_wait, thanks, acknowledge","informal"),
("T37","driver","Should I turn the AC off?","ask_ac, acknowledge","clean"),
("T38","driver","drop pt is near the flyover right?","confirm_dropoff","informal"),
("T39","driver","will reach in few mins","acknowledge, thanks","informal"),
("T40","driver","Are you paying cash or card?","cash_payment","clean"),
("T41","driver","Hope you had a pleasant ride","thanks, goodbye","clean"),
("T42","driver","aage kahan rokun","stop_next_corner, confirm_dropoff","hinglish"),
("T43","driver","Traffic is heavy, a little late","acknowledge, ask_how_far, please_wait","clean"),
("T44","driver","trip ended, thank you","thanks, goodbye","informal"),
("T45","driver","only 5 mins free waiting, then charges apply","sorry_late, coming_now, please_wait","informal"),
("T46","driver","pickup point pe ho ya nahi","at_pickup_point, coming_now","hinglish"),
("T47","driver","Which side of the road are you on?","at_pickup_point, cant_see_you, can_see_you","clean"),
("T48","driver","plz share ur live loc","at_pickup_point, cant_see_you, please_wait","informal"),
("T49","driver","Namaste madam","greeting","hinglish"),
("T50","driver","Do you want me to go slower?","ask_slow_down, acknowledge","clean"),
("T51","driver","I can't wait much longer","sorry_late, coming_now, please_wait","clean"),
("T52","driver","Road is closed ahead, can you walk to the corner?","acknowledge, coming_now, stop_next_corner","clean"),
("T53","driver","arrived at ur drop, pls check","confirm_dropoff, thanks, acknowledge","informal"),
("T54","driver","ride khatam, dhanyawad","thanks, goodbye","hinglish"),
("T55","driver","is it gate 3 where u r standing","at_pickup_point, coming_now","informal"),
("T56","driver","ok?","none","informal"),
("T57","driver","Did you watch the movie?","none","clean"),
("T58","driver","random words purple table","none","informal"),
("T59","driver","lol","none","informal"),
("T60","driver","What is the capital of France?","none","clean"),
]

test = pd.DataFrame(test_rows, columns=["id","sender_role","text","acceptable_intents","style"])
test["acceptable_intents"] = test["acceptable_intents"].apply(
    lambda s: "none" if s.strip() == "none" else "|".join(x.strip() for x in s.split(",")))
test["answerable"] = test["acceptable_intents"] != "none"

# Checks: valid intents, unique ids, no text shared with dev or the bank
bank = pd.read_csv(os.path.join(CONFIG["data_dir"], "reply_bank.csv"))
dev = pd.read_csv(os.path.join(CONFIG["data_dir"], "dev_messages.csv"))
reply_role = {"rider": "driver", "driver": "rider"}
for _, r in test[test["answerable"]].iterrows():
    valid = set(bank[bank["role"] == reply_role[r["sender_role"]]]["intent"])
    bad = [i for i in r["acceptable_intents"].split("|") if i not in valid]
    assert not bad, f"{r['id']}: unknown intents {bad}"
assert test["id"].is_unique
norm = lambda s: s.str.lower().str.strip()
assert not set(norm(test["text"])) & set(norm(dev["text"])), "Test text also in dev"
assert not set(norm(test["text"])) & set(norm(bank["text"])), "Test text also in bank"

# Save and show counts
test.to_csv(os.path.join(CONFIG["data_dir"], "test_messages.csv"), index=False)
print("Saved test messages:", len(test))
print("\nPer sender role:\n", test["sender_role"].value_counts())
print("\nPer style:\n", test["style"].value_counts())
print("\nAnswerable or not:\n", test["answerable"].value_counts())
print("\nSender role x answerable:\n", pd.crosstab(test["sender_role"], test["answerable"]))

Saved test messages: 60

Per sender role:
 sender_role
rider     30
driver    30
Name: count, dtype: int64

Per style:
 style
informal    27
clean       23
hinglish    10
Name: count, dtype: int64

Answerable or not:
 answerable
True     50
False    10
Name: count, dtype: int64

Sender role x answerable:
 answerable   False  True 
sender_role              
driver           5     25
rider            5     25


In [5]:
from sentence_transformers import SentenceTransformer

# Load the model on CPU (same as the server)
model = SentenceTransformer(CONFIG["model_name"], device="cpu")

# Encode the bank once; normalized vectors so dot product = cosine similarity
bank = pd.read_csv(os.path.join(CONFIG["data_dir"], "reply_bank.csv"))
bank_emb = model.encode(bank["text"].tolist(), normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=False)

print("Bank embeddings shape:", bank_emb.shape)
print("Dtype:", bank_emb.dtype)
print("Vector length check (should be ~1.0):", float(np.linalg.norm(bank_emb[0])))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Bank embeddings shape: (80, 384)
Dtype: float32
Vector length check (should be ~1.0): 1.0


In [6]:
# Temporary threshold (we choose the real one in step 7 using dev messages)
TEMP_THRESHOLD = 0.40
MAX_CHARS = 500

# Precompute row positions of each role in the bank (for speed)
role_rows = {r: np.where(bank["role"].values == r)[0] for r in ["driver", "rider"]}

def suggest_replies(message, recipient_role, flagged=False, k=3, threshold=TEMP_THRESHOLD):
    """recipient_role = person who RECEIVES the message (and will send the reply)."""
    # Moderation flag -> no suggestions
    if flagged:
        return []
    # Bad input -> no suggestions
    if not isinstance(message, str) or recipient_role not in role_rows:
        return []
    message = message.strip()[:MAX_CHARS]   # cut very long text
    if not message:
        return []
    # Score the message against replies of the replying role only
    rows = role_rows[recipient_role]
    q = model.encode([message], normalize_embeddings=True, convert_to_numpy=True,
                     show_progress_bar=False)[0]
    scores = bank_emb[rows] @ q
    # Best reply per intent
    best = {}
    for pos, s in zip(rows, scores):
        intent = bank.at[pos, "intent"]
        if intent not in best or s > best[intent][0]:
            best[intent] = (float(s), pos)
    ranked = sorted(best.values(), key=lambda x: x[0], reverse=True)
    # Nothing good enough -> show nothing
    if not ranked or ranked[0][0] < threshold:
        return []
    # Top k different intents (only those above threshold)
    out = []
    for s, pos in ranked[:k]:
        if s >= threshold:
            out.append({"id": bank.at[pos, "id"], "intent": bank.at[pos, "intent"],
                        "text": bank.at[pos, "text"], "score": round(s, 3)})
    return out

# Quick try-out on a few dev messages (rider message -> driver replies, and so on)
dev = pd.read_csv(os.path.join(CONFIG["data_dir"], "dev_messages.csv"))
for mid in ["DV01", "DV06", "DV19", "DV22", "DV28"]:
    r = dev[dev["id"] == mid].iloc[0]
    recipient = "driver" if r["sender_role"] == "rider" else "rider"
    print(mid, "|", r["text"], "| acceptable:", r["acceptable_intents"])
    for s in suggest_replies(r["text"], recipient):
        print("   ", s["id"], s["intent"], "-", s["text"], s["score"])

# Edge cases
print("\nEmpty text:", suggest_replies("   ", "driver"))
print("Not a string:", suggest_replies(None, "driver"))
print("Wrong role:", suggest_replies("hello", "admin"))
print("Flagged:", suggest_replies("hello", "driver", flagged=True))
print("Very long text:", len(suggest_replies("where are you " * 200, "driver")), "suggestions, no crash")

DV01 | hey driver, where r u now | acceptable: on_my_way|few_minutes_away|stuck_in_traffic
    D02 greeting - Hi, I'm your driver. 0.721
    D10 arrived_pickup - Reached your pickup location. 0.412
DV06 | Can I pay by cash at the end of the trip? | acceptable: cash_payment
    D37 cash_payment - Yes, cash is fine. 0.564
DV19 | hmm | acceptable: none
DV22 | I've reached the spot, where are you? | acceptable: at_pickup_point|coming_now|please_wait
    R14 cant_see_you - I can't see you. Where are you? 0.57
    R10 at_pickup_point - I'm at the pinned location. 0.556
    R33 ask_how_far - How far are you? 0.441
DV28 | is this the corect drop point | acceptable: confirm_dropoff
    R24 confirm_dropoff - Yes, drop-off is as per the app. 0.414

Empty text: []
Not a string: []
Wrong role: []
Flagged: []
Very long text: 0 suggestions, no crash


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit TF-IDF on the bank texts only (no outside data)
tfidf = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
bank_tfidf = tfidf.fit_transform(bank["text"].tolist())   # rows are L2-normalized

BASE_TEMP_THRESHOLD = 0.20   # temporary; TF-IDF scores have a different scale

def suggest_replies_tfidf(message, recipient_role, flagged=False, k=3, threshold=BASE_TEMP_THRESHOLD):
    # Same rules as suggest_replies: flag, role filter, best per intent, top k intents, threshold
    if flagged or not isinstance(message, str) or recipient_role not in role_rows:
        return []
    message = message.strip()[:MAX_CHARS]
    if not message:
        return []
    rows = role_rows[recipient_role]
    q = tfidf.transform([message])
    scores = (bank_tfidf[rows] @ q.T).toarray().ravel()   # cosine (vectors are normalized)
    best = {}
    for pos, s in zip(rows, scores):
        intent = bank.at[pos, "intent"]
        if intent not in best or s > best[intent][0]:
            best[intent] = (float(s), pos)
    ranked = sorted(best.values(), key=lambda x: x[0], reverse=True)
    if not ranked or ranked[0][0] < threshold:
        return []
    return [{"id": bank.at[pos, "id"], "intent": bank.at[pos, "intent"],
             "text": bank.at[pos, "text"], "score": round(s, 3)}
            for s, pos in ranked[:k] if s >= threshold]

# Try the same dev messages as before
for mid in ["DV01", "DV06", "DV19", "DV22", "DV28"]:
    r = dev[dev["id"] == mid].iloc[0]
    recipient = "driver" if r["sender_role"] == "rider" else "rider"
    print(mid, "|", r["text"], "| acceptable:", r["acceptable_intents"])
    for s in suggest_replies_tfidf(r["text"], recipient):
        print("   ", s["id"], s["intent"], "-", s["text"], s["score"])

# Edge cases
print("\nEmpty text:", suggest_replies_tfidf("   ", "driver"))
print("Not a string:", suggest_replies_tfidf(None, "driver"))
print("Flagged:", suggest_replies_tfidf("hello", "driver", flagged=True))

DV01 | hey driver, where r u now | acceptable: on_my_way|few_minutes_away|stuck_in_traffic
    D02 greeting - Hi, I'm your driver. 0.314
    D15 ask_location - Where exactly are you? 0.205
DV06 | Can I pay by cash at the end of the trip? | acceptable: cash_payment
    D23 cant_find_you - I can't see you at the pickup. 0.329
    D09 arrived_pickup - I'm here at the pickup point. 0.261
    D11 waiting_at_pickup - I'm waiting at the pickup point. 0.259
DV19 | hmm | acceptable: none
DV22 | I've reached the spot, where are you? | acceptable: at_pickup_point|coming_now|please_wait
    R15 cant_see_you - Where are you exactly? 0.671
    R33 ask_how_far - How far are you? 0.32
DV28 | is this the corect drop point | acceptable: confirm_dropoff
    R24 confirm_dropoff - Yes, drop-off is as per the app. 0.305
    R09 at_pickup_point - I'm at the pickup point on the map. 0.258
    R35 come_to_pickup - Please come to the pickup point. 0.21

Empty text: []
Not a string: []
Flagged: []


In [9]:
# Evaluate one method on the dev messages at one threshold
def run_eval(fn, df, thr):
    top1 = top3 = none_ok = 0
    n_ans = int(df["answerable"].sum()); n_none = len(df) - n_ans
    for _, r in df.iterrows():
        recipient = "driver" if r["sender_role"] == "rider" else "rider"
        sugg = fn(r["text"], recipient, threshold=thr)
        if r["answerable"]:
            ok = set(r["acceptable_intents"].split("|"))
            top1 += bool(sugg) and sugg[0]["intent"] in ok
            top3 += any(s["intent"] in ok for s in sugg)
        else:
            none_ok += (len(sugg) == 0)
    return top1 / n_ans, top3 / n_ans, none_ok / n_none

# Threshold table
rows = []
for thr in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    m = run_eval(suggest_replies, dev, thr)
    b = run_eval(suggest_replies_tfidf, dev, thr)
    rows.append([thr, *[round(x, 3) for x in m], *[round(x, 3) for x in b]])
table = pd.DataFrame(rows, columns=["thr", "MiniLM_top1", "MiniLM_top3", "MiniLM_none_ok",
                                    "TFIDF_top1", "TFIDF_top3", "TFIDF_none_ok"])
print("Dev: 36 answerable, 4 unanswerable")
print(table.to_string(index=False))

# Threshold-free failures: answerable messages with NO acceptable intent in the top 3
def show_misses(fn, name):
    print(f"\n{name}: answerable dev messages with no acceptable intent in top 3 (no threshold)")
    for _, r in dev[dev["answerable"]].iterrows():
        recipient = "driver" if r["sender_role"] == "rider" else "rider"
        sugg = fn(r["text"], recipient, threshold=-1.0)
        ok = set(r["acceptable_intents"].split("|"))
        if not any(s["intent"] in ok for s in sugg):
            print(r["id"], "|", r["text"], "| want:", r["acceptable_intents"],
                  "| got:", [(s["intent"], s["score"]) for s in sugg])
show_misses(suggest_replies, "MiniLM")
show_misses(suggest_replies_tfidf, "TF-IDF")

# Best score of the unanswerable dev messages (the threshold must be above these)
print("\nMiniLM best score on unanswerable dev messages:")
for _, r in dev[~dev["answerable"]].iterrows():
    recipient = "driver" if r["sender_role"] == "rider" else "rider"
    s = suggest_replies(r["text"], recipient, threshold=-1.0)
    print(r["id"], "|", r["text"], "|", s[0]["score"] if s else None)

Dev: 36 answerable, 4 unanswerable
 thr  MiniLM_top1  MiniLM_top3  MiniLM_none_ok  TFIDF_top1  TFIDF_top3  TFIDF_none_ok
0.20        0.444        0.722            0.50       0.361       0.389           0.75
0.25        0.444        0.694            0.50       0.333       0.361           0.75
0.30        0.444        0.667            0.75       0.306       0.306           0.75
0.35        0.417        0.611            0.75       0.194       0.194           0.75
0.40        0.389        0.472            1.00       0.194       0.194           1.00
0.45        0.222        0.278            1.00       0.167       0.167           1.00
0.50        0.194        0.222            1.00       0.139       0.139           1.00
0.55        0.194        0.222            1.00       0.139       0.139           1.00
0.60        0.083        0.083            1.00       0.083       0.083           1.00

MiniLM: answerable dev messages with no acceptable intent in top 3 (no threshold)
DV08 | i cant see ur c

In [10]:
# Apply approved bank changes (safe to run twice: rows are replaced by id)
bank_path = os.path.join(CONFIG["data_dir"], "reply_bank.csv")
old_path  = os.path.join(CONFIG["data_dir"], "reply_bank_v1.csv")
cur = pd.read_csv(bank_path)
if not os.path.exists(old_path):          # keep the old bank, never overwrite it
    cur.to_csv(old_path, index=False)

changes = pd.DataFrame([
    ("D02","driver","greeting","Hi there!"),
    ("D41","driver","greeting","Good evening!"),
    ("D42","driver","acknowledge","Sure, no problem."),
    ("D43","driver","acknowledge","Okay, see you soon."),
    ("D44","driver","acknowledge","No problem, take your time."),
    ("D45","driver","waiting_at_pickup","Sure, I'll wait there."),
    ("R02","rider","greeting","Hi there!"),
    ("R41","rider","greeting","Good evening!"),
    ("R42","rider","acknowledge","Okay, see you soon."),
    ("R43","rider","acknowledge","Okay, thanks for letting me know."),
], columns=cur.columns[:4])

# Replace rows with the same id, add new ones, keep the order by id
new_bank = pd.concat([cur[~cur["id"].isin(changes["id"])], changes])
new_bank = new_bank.sort_values("id").reset_index(drop=True)
new_bank.to_csv(bank_path, index=False)

# Rebuild everything that depends on the bank (the functions read these globals)
bank = new_bank
bank_emb = model.encode(bank["text"].tolist(), normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=False)
role_rows = {r: np.where(bank["role"].values == r)[0] for r in ["driver", "rider"]}
tfidf = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
bank_tfidf = tfidf.fit_transform(bank["text"].tolist())

print("Bank size:", len(bank), "| embeddings:", bank_emb.shape)
print(bank.groupby("role")["intent"].agg(replies="count", intents="nunique"))

# Re-run the dev evaluation (same functions as cell 7)
rows = []
for thr in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    m = run_eval(suggest_replies, dev, thr)
    b = run_eval(suggest_replies_tfidf, dev, thr)
    rows.append([thr, *[round(x, 3) for x in m], *[round(x, 3) for x in b]])
print(pd.DataFrame(rows, columns=["thr","MiniLM_top1","MiniLM_top3","MiniLM_none_ok",
                                  "TFIDF_top1","TFIDF_top3","TFIDF_none_ok"]).to_string(index=False))

show_misses(suggest_replies, "MiniLM")
show_misses(suggest_replies_tfidf, "TF-IDF")

print("\nMiniLM best score on unanswerable dev messages:")
for _, r in dev[~dev["answerable"]].iterrows():
    recipient = "driver" if r["sender_role"] == "rider" else "rider"
    s = suggest_replies(r["text"], recipient, threshold=-1.0)
    print(r["id"], "|", r["text"], "|", s[0]["score"] if s else None)

Bank size: 88 | embeddings: (88, 384)
        replies  intents
role                    
driver       45       17
rider        43       17
 thr  MiniLM_top1  MiniLM_top3  MiniLM_none_ok  TFIDF_top1  TFIDF_top3  TFIDF_none_ok
0.20        0.556        0.861            0.50       0.417       0.472           0.75
0.25        0.556        0.806            0.50       0.417       0.444           0.75
0.30        0.556        0.778            0.75       0.361       0.361           0.75
0.35        0.528        0.750            0.75       0.250       0.250           0.75
0.40        0.444        0.556            1.00       0.250       0.250           1.00
0.45        0.278        0.333            1.00       0.194       0.194           1.00
0.50        0.250        0.278            1.00       0.194       0.194           1.00
0.55        0.250        0.278            1.00       0.194       0.194           1.00
0.60        0.167        0.167            1.00       0.139       0.139           1.00

M

In [11]:
# FINAL TEST: run once. MiniLM threshold 0.30, TF-IDF threshold 0.20 (both chosen on dev)
test = pd.read_csv(os.path.join(CONFIG["data_dir"], "test_messages.csv"))
if "answerable" not in test.columns:
    test["answerable"] = test["acceptable_intents"].str.strip().str.lower() != "none"

def eval_one(fn, thr):
    out = []
    for _, r in test.iterrows():
        recipient = "driver" if r["sender_role"] == "rider" else "rider"
        sugg = fn(r["text"], recipient, threshold=thr)
        if r["answerable"]:
            ok = set(r["acceptable_intents"].split("|"))
            t1 = bool(sugg) and sugg[0]["intent"] in ok
            t3 = any(s["intent"] in ok for s in sugg)
        else:
            t1 = t3 = (len(sugg) == 0)      # correct = no suggestion
        out.append((r["id"], r["text"], r["style"], bool(r["answerable"]), t1, t3,
                    [(s["intent"], s["score"]) for s in sugg]))
    return pd.DataFrame(out, columns=["id","text","style","answerable","top1","top3","got"])

results = {"MiniLM": eval_one(suggest_replies, 0.30),
           "TF-IDF": eval_one(suggest_replies_tfidf, 0.20)}

summary = {}
for name, d in results.items():
    ans, non = d[d["answerable"]], d[~d["answerable"]]
    print(f"\n=== {name} ===")
    print(f"Answerable ({len(ans)}): top-1 = {ans['top1'].mean():.3f} | top-3 = {ans['top3'].mean():.3f}")
    print(f"Unanswerable ({len(non)}): correct no-suggestion = {non['top1'].mean():.3f}")
    by_style = ans.groupby("style").agg(n=("top1","size"), top1=("top1","mean"), top3=("top3","mean")).round(3)
    print("By style (answerable only):"); print(by_style)
    print("Unanswerable correct by style:"); print(non.groupby("style")["top1"].agg(["size","sum"]))
    print("Failed messages:")
    for _, r in d[~d["top3"]].iterrows():
        print("  ", r["id"], "|", r["text"], "| answerable:", r["answerable"], "| got:", r["got"])
    summary[name] = {"top1": round(ans["top1"].mean(), 3), "top3": round(ans["top3"].mean(), 3),
                     "none_ok": round(non["top1"].mean(), 3)}

# Save the final results (new file, nothing overwritten)
with open(os.path.join(CONFIG["data_dir"], "final_test_results.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("\nSaved:", summary)


=== MiniLM ===
Answerable (50): top-1 = 0.480 | top-3 = 0.660
Unanswerable (10): correct no-suggestion = 0.700
By style (answerable only):
           n   top1   top3
style                     
clean     18  0.778  0.889
hinglish  10  0.100  0.200
informal  22  0.409  0.682
Unanswerable correct by style:
          size  sum
style              
clean        5    5
informal     5    2
Failed messages:
   T05 | bhaiya kitni door ho | answerable: True | got: []
   T07 | thnks, smooth ride | answerable: True | got: [('stuck_in_traffic', 0.303)]
   T10 | main gate ke paas khada hu | answerable: True | got: []
   T16 | bhai gaadi kahan hai | answerable: True | got: []
   T19 | my pin is wrong, im 50m ahead | answerable: True | got: [('stuck_in_traffic', 0.309)]
   T20 | drop me at the bus stand pls | answerable: True | got: [('cant_find_you', 0.39), ('stuck_in_traffic', 0.334), ('please_wait', 0.322)]
   T21 | r u still there? | answerable: True | got: [('acknowledge', 0.386), ('stuck_in_traf

In [12]:
# Step 9 (optional): compare with the larger model all-mpnet-base-v2 (same rules, threshold 0.30)
mp = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device="cpu")
mp_emb = mp.encode(bank["text"].tolist(), normalize_embeddings=True,
                   convert_to_numpy=True, show_progress_bar=False)

# Same logic as suggest_replies, but with any model
def make_fn(m, emb):
    def fn(message, recipient_role, flagged=False, k=3, threshold=0.30):
        if flagged or not isinstance(message, str) or recipient_role not in role_rows:
            return []
        message = message.strip()[:MAX_CHARS]
        if not message:
            return []
        rows = role_rows[recipient_role]
        q = m.encode([message], normalize_embeddings=True, convert_to_numpy=True,
                     show_progress_bar=False)[0]
        scores = emb[rows] @ q
        best = {}
        for pos, s in zip(rows, scores):
            it = bank.at[pos, "intent"]
            if it not in best or s > best[it][0]:
                best[it] = (float(s), pos)
        ranked = sorted(best.values(), key=lambda x: x[0], reverse=True)
        if not ranked or ranked[0][0] < threshold:
            return []
        return [{"id": bank.at[p, "id"], "intent": bank.at[p, "intent"],
                 "text": bank.at[p, "text"], "score": round(s, 3)}
                for s, p in ranked[:k] if s >= threshold]
    return fn

mp_fn = make_fn(mp, mp_emb)
d = eval_one(mp_fn, 0.30)
ans, non = d[d["answerable"]], d[~d["answerable"]]
print(f"mpnet answerable ({len(ans)}): top-1 = {ans['top1'].mean():.3f} | top-3 = {ans['top3'].mean():.3f}")
print(f"mpnet correct no-suggestion ({len(non)}): {non['top1'].mean():.3f}")
print(ans.groupby("style").agg(n=("top1","size"), top1=("top1","mean"), top3=("top3","mean")).round(3))

# Speed on CPU: 100 single-message runs for each model
msgs = test["text"].tolist()
def time_fn(fn):
    times = []
    for i in range(100):
        t = time.perf_counter(); fn(msgs[i % len(msgs)], "driver"); times.append((time.perf_counter() - t) * 1000)
    return np.mean(times), np.percentile(times, 95)
for name, fn in [("MiniLM", suggest_replies), ("mpnet", mp_fn)]:
    a, p = time_fn(fn)
    print(f"{name}: average {a:.1f} ms | 95th percentile {p:.1f} ms")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

mpnet answerable (50): top-1 = 0.460 | top-3 = 0.620
mpnet correct no-suggestion (10): 0.900
           n   top1   top3
style                     
clean     18  0.556  0.667
hinglish  10  0.100  0.200
informal  22  0.545  0.773
MiniLM: average 22.1 ms | 95th percentile 27.9 ms
mpnet: average 105.7 ms | 95th percentile 137.1 ms


In [13]:
# Step 10: speed and size on CPU (MiniLM, threshold 0.30)
import shutil
torch.set_num_threads(os.cpu_count() or 1)
print("CPU cores:", os.cpu_count(), "| torch threads:", torch.get_num_threads(), "| device:", model.device)

# Warm-up (first calls are slower), then 200 timed runs, one message at a time
msgs = test["text"].tolist() + dev["text"].tolist()
for i in range(5):
    suggest_replies(msgs[i], "driver", threshold=0.30)
times = []
for i in range(200):
    t = time.perf_counter()
    suggest_replies(msgs[i % len(msgs)], "driver" if i % 2 else "rider", threshold=0.30)
    times.append((time.perf_counter() - t) * 1000)
print(f"Runs: {len(times)} | average {np.mean(times):.1f} ms | median {np.median(times):.1f} ms | "
      f"95th percentile {np.percentile(times, 95):.1f} ms | max {np.max(times):.1f} ms")

# Model size on disk: save to a NEW folder (reused in step 11)
model_dir = os.path.join(CONFIG["deploy_dir"], "minilm_model")
if os.path.exists(model_dir):
    print("Folder already exists, not overwriting:", model_dir)
else:
    model.save(model_dir)
total = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(model_dir) for f in fs)
print(f"Model folder size: {total / 1024**2:.1f} MB")

# Bank size
print("Bank replies:", len(bank), "| embeddings:", bank_emb.shape,
      f"| embeddings in memory: {bank_emb.nbytes / 1024:.1f} KB",
      f"| bank CSV: {os.path.getsize(bank_path) / 1024:.1f} KB")

CPU cores: 2 | torch threads: 2 | device: cpu
Runs: 200 | average 68.0 ms | median 21.2 ms | 95th percentile 294.7 ms | max 781.3 ms


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model folder size: 87.3 MB
Bank replies: 88 | embeddings: (88, 384) | embeddings in memory: 132.0 KB | bank CSV: 4.1 KB


In [14]:
# Step 11: save deployable files in ONE new folder, reload from disk, and check
import shutil, datetime

# New folder name (never overwrite an existing one)
pkg = os.path.join(CONFIG["deploy_dir"], "smartreply_v1"); n = 1
while os.path.exists(pkg):
    n += 1; pkg = os.path.join(CONFIG["deploy_dir"], f"smartreply_v{n}")
os.makedirs(pkg)

# Model (offline copy), bank, bank embeddings (same row order), dev and test CSVs
shutil.copytree(model_dir, os.path.join(pkg, "model"))
bank[["id","role","intent","text"]].to_csv(os.path.join(pkg, "reply_bank.csv"), index=False)
np.save(os.path.join(pkg, "bank_embeddings.npy"), bank_emb)
for f in ["dev_messages.csv", "test_messages.csv"]:
    shutil.copy(os.path.join(CONFIG["data_dir"], f), os.path.join(pkg, f))

# Config
config = {"model_name": CONFIG["model_name"], "threshold": 0.30, "k": 3,
          "max_chars": MAX_CHARS,
          "intent_diversity_rule": "best reply per intent, then top-k different intents; only replies of the replying role; empty list if best score < threshold; flagged=True returns empty list",
          "sentence_transformers_version": sentence_transformers.__version__,
          "created": datetime.date.today().isoformat()}
json.dump(config, open(os.path.join(pkg, "replies_config.json"), "w"), indent=2)

# Golden cases: 10 messages with the ids the function gives now
gold_ids = ["DV02","DV06","DV12","DV13","DV20","DV21","DV28","DV30","DV37","T06"]
allmsg = pd.concat([dev, test])
golden = []
for _, r in allmsg[allmsg["id"].isin(gold_ids)].iterrows():
    rec = "driver" if r["sender_role"] == "rider" else "rider"
    out = suggest_replies(r["text"], rec, threshold=0.30)
    golden.append({"message": r["text"], "recipient_role": rec,
                   "expected_ids": [s["id"] for s in out], "expected_scores": [s["score"] for s in out]})
json.dump(golden, open(os.path.join(pkg, "golden_test_cases.json"), "w"), indent=2)

# Metrics (final test, by style, speed, size, mpnet numbers from step 9)
metrics = {"final_test": json.load(open(os.path.join(CONFIG["data_dir"], "final_test_results.json")))}
for name, dd in results.items():
    a = dd[dd["answerable"]]
    metrics.setdefault("by_style", {})[name] = {
        s: {"n": int(g["top1"].size), "top1": round(float(g["top1"].mean()), 3),
            "top3": round(float(g["top3"].mean()), 3)} for s, g in a.groupby("style")}
metrics["speed_ms_minilm_step10"] = {"runs": len(times), "average": round(float(np.mean(times)), 1),
    "median": round(float(np.median(times)), 1), "p95": round(float(np.percentile(times, 95)), 1)}
metrics["model_size_mb"] = round(total / 1024**2, 1)
metrics["mpnet_step9"] = {"top1": 0.46, "top3": 0.62, "none_ok": 0.9, "avg_ms": 105.7, "p95_ms": 137.1}
json.dump(metrics, open(os.path.join(pkg, "replies_metrics.json"), "w"), indent=2)

# ---- Reload everything from disk and check ----
cfg2 = json.load(open(os.path.join(pkg, "replies_config.json")))
bank2 = pd.read_csv(os.path.join(pkg, "reply_bank.csv"))
emb2 = np.load(os.path.join(pkg, "bank_embeddings.npy"))
model2 = SentenceTransformer(os.path.join(pkg, "model"), device="cpu")
rr2 = {r: np.where(bank2["role"].values == r)[0] for r in ["driver", "rider"]}

def suggest2(message, recipient_role, flagged=False, k=None, threshold=None):
    k = k or cfg2["k"]; threshold = cfg2["threshold"] if threshold is None else threshold
    if flagged or not isinstance(message, str) or recipient_role not in rr2:
        return []
    message = message.strip()[:cfg2["max_chars"]]
    if not message:
        return []
    rows = rr2[recipient_role]
    q = model2.encode([message], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)[0]
    sc = emb2[rows] @ q
    best = {}
    for pos, s in zip(rows, sc):
        it = bank2.at[pos, "intent"]
        if it not in best or s > best[it][0]:
            best[it] = (float(s), pos)
    ranked = sorted(best.values(), key=lambda x: x[0], reverse=True)
    if not ranked or ranked[0][0] < threshold:
        return []
    return [{"id": bank2.at[p, "id"], "intent": bank2.at[p, "intent"], "text": bank2.at[p, "text"],
             "score": round(s, 3)} for s, p in ranked[:k] if s >= threshold]

g2 = json.load(open(os.path.join(pkg, "golden_test_cases.json")))
bad = 0
for g in g2:
    out = suggest2(g["message"], g["recipient_role"])
    same = [s["id"] for s in out] == g["expected_ids"] and \
           all(abs(s["score"] - e) < 0.002 for s, e in zip(out, g["expected_scores"]))
    bad += (not same)
    print("OK " if same else "DIFF", g["message"], "->", [s["id"] for s in out])
print(f"\nGolden cases: {len(g2) - bad}/{len(g2)} match")
print("Flagged returns empty list:", suggest2("where are you", "driver", flagged=True) == [])
print("Same order check (bank vs embeddings):", len(bank2) == emb2.shape[0])
print("Files in package:", sorted(os.listdir(pkg)))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

OK  hii -> ['D02']
OK  Can I pay by cash at the end of the trip? -> ['D37']
OK  which car number are u in -> ['D21']
OK  Good evening -> ['D41', 'D40', 'D05']
OK  what is the score of the match -> []
OK  Good morning ma'am, driver here -> ['R41', 'R11', 'R16']
OK  is this the corect drop point -> ['R24', 'R09', 'R30']
OK  Payment will be cash or online? -> ['R29']
OK  Are you standing at the spot shown on the map? -> ['R09', 'R13', 'R14']
OK  I only have cash, is that ok? -> ['D37']

Golden cases: 10/10 match
Flagged returns empty list: True
Same order check (bank vs embeddings): True
Files in package: ['bank_embeddings.npy', 'dev_messages.csv', 'golden_test_cases.json', 'model', 'replies_config.json', 'replies_metrics.json', 'reply_bank.csv', 'test_messages.csv']


In [15]:
# Step 12: numbers for my report (read from saved files and data, nothing typed by hand)
m = json.load(open(os.path.join(pkg, "replies_metrics.json")))
c = json.load(open(os.path.join(pkg, "replies_config.json")))
b = pd.read_csv(os.path.join(pkg, "reply_bank.csv"))
dv = pd.read_csv(os.path.join(pkg, "dev_messages.csv"))
ts = pd.read_csv(os.path.join(pkg, "test_messages.csv"))

print("=== NUMBERS FOR MY REPORT ===")
print("Model:", c["model_name"], "| sentence-transformers", c["sentence_transformers_version"])
print("\nReply bank:", len(b), "replies |", b.groupby("role").size().to_dict(),
      "| intents per role:", b.groupby("role")["intent"].nunique().to_dict(),
      "| total different intents:", b["intent"].nunique())
print("Dev messages:", len(dv), "| Test messages:", len(ts),
      "(answerable 50, unanswerable 10)")
print("Chosen threshold (MiniLM):", c["threshold"], "| k:", c["k"], "| TF-IDF threshold: 0.20")

print("\nFINAL TEST (50 answerable, 10 unanswerable)")
for name in ["MiniLM", "TF-IDF"]:
    r = m["final_test"][name]
    print(f"  {name}: top-1 {r['top1']:.2f} | top-3 {r['top3']:.2f} | correct no-suggestion {r['none_ok']:.2f}")

print("\nBY STYLE (answerable only: n, top-1, top-3)")
for name in ["MiniLM", "TF-IDF"]:
    for s, v in m["by_style"][name].items():
        print(f"  {name:7s} {s:9s} n={v['n']:2d} top-1 {v['top1']:.2f} top-3 {v['top3']:.2f}")

sp = m["speed_ms_minilm_step10"]
print(f"\nSpeed on a 2-core Colab CPU ({sp['runs']} runs): average {sp['average']} ms | "
      f"median {sp['median']} ms | 95th percentile {sp['p95']} ms")
print("Model size on disk:", m["model_size_mb"], "MB | bank embeddings: 132 KB")
print("mpnet (optional comparison): top-1 0.46, top-3 0.62, no-suggestion 0.90, "
      "average 105.7 ms, 95th percentile 137.1 ms")

=== NUMBERS FOR MY REPORT ===
Model: sentence-transformers/all-MiniLM-L6-v2 | sentence-transformers 5.7.0

Reply bank: 88 replies | {'driver': 45, 'rider': 43} | intents per role: {'driver': 17, 'rider': 17} | total different intents: 27
Dev messages: 40 | Test messages: 60 (answerable 50, unanswerable 10)
Chosen threshold (MiniLM): 0.3 | k: 3 | TF-IDF threshold: 0.20

FINAL TEST (50 answerable, 10 unanswerable)
  MiniLM: top-1 0.48 | top-3 0.66 | correct no-suggestion 0.70
  TF-IDF: top-1 0.42 | top-3 0.48 | correct no-suggestion 0.40

BY STYLE (answerable only: n, top-1, top-3)
  MiniLM  clean     n=18 top-1 0.78 top-3 0.89
  MiniLM  hinglish  n=10 top-1 0.10 top-3 0.20
  MiniLM  informal  n=22 top-1 0.41 top-3 0.68
  TF-IDF  clean     n=18 top-1 0.56 top-3 0.67
  TF-IDF  hinglish  n=10 top-1 0.30 top-3 0.30
  TF-IDF  informal  n=22 top-1 0.36 top-3 0.41

Speed on a 2-core Colab CPU (200 runs): average 68.0 ms | median 21.2 ms | 95th percentile 294.7 ms
Model size on disk: 87.3 MB | 

In [16]:
# Zip the deployable folder (+ a ready-to-use Python file) and download it
import zipfile
from google.colab import files

# 1) Small standalone module for your project (reads only files from the folder, works offline)
code = '''"""Smart replies for QuickPull. Usage:
    from smart_reply import SmartReplier
    sr = SmartReplier("path/to/smartreply_v1")
    sr.suggest("where are you?", recipient_role="driver", flagged=False)
recipient_role = the person who RECEIVES the message and will send the reply.
"""
import os, csv, json
import numpy as np
from sentence_transformers import SentenceTransformer


class SmartReplier:
    def __init__(self, folder):
        self.cfg = json.load(open(os.path.join(folder, "replies_config.json")))
        with open(os.path.join(folder, "reply_bank.csv"), newline="", encoding="utf-8") as f:
            self.bank = list(csv.DictReader(f))
        self.emb = np.load(os.path.join(folder, "bank_embeddings.npy"))
        self.model = SentenceTransformer(os.path.join(folder, "model"), device="cpu")
        self.rows = {r: [i for i, b in enumerate(self.bank) if b["role"] == r]
                     for r in ("driver", "rider")}

    def suggest(self, message, recipient_role, flagged=False, k=None, threshold=None):
        k = k or self.cfg["k"]
        threshold = self.cfg["threshold"] if threshold is None else threshold
        if flagged or not isinstance(message, str) or recipient_role not in self.rows:
            return []
        message = message.strip()[: self.cfg["max_chars"]]
        if not message:
            return []
        idx = self.rows[recipient_role]
        q = self.model.encode([message], normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False)[0]
        scores = self.emb[idx] @ q
        best = {}  # best reply per intent
        for i, s in zip(idx, scores):
            intent = self.bank[i]["intent"]
            if intent not in best or s > best[intent][0]:
                best[intent] = (float(s), i)
        ranked = sorted(best.values(), key=lambda x: x[0], reverse=True)
        if not ranked or ranked[0][0] < threshold:
            return []
        return [{"id": self.bank[i]["id"], "intent": self.bank[i]["intent"],
                 "text": self.bank[i]["text"], "score": round(s, 3)}
                for s, i in ranked[:k] if s >= threshold]


if __name__ == "__main__":
    sr = SmartReplier(os.path.dirname(os.path.abspath(__file__)))
    print(sr.suggest("where are you?", "driver"))
'''
sm_path = os.path.join(pkg, "smart_reply.py")
if not os.path.exists(sm_path):
    open(sm_path, "w", encoding="utf-8").write(code)

req_path = os.path.join(pkg, "requirements.txt")
if not os.path.exists(req_path):
    open(req_path, "w").write(f"sentence-transformers=={sentence_transformers.__version__}\nnumpy\n")

# 2) Quick check: the new module gives the same answer as the notebook function
import importlib.util
spec = importlib.util.spec_from_file_location("smart_reply", sm_path)
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
sr = mod.SmartReplier(pkg)
test_msg = "which car number are u in"
print("Module result :", [s["id"] for s in sr.suggest(test_msg, "driver")])
print("Notebook result:", [s["id"] for s in suggest_replies(test_msg, "driver", threshold=0.30)])

# 3) Zip (new name, never overwrite) and download
zip_base = os.path.join(CONFIG["base_dir"], "quickpull_smartreply_package"); n = 1
while os.path.exists(zip_base + ".zip"):
    n += 1; zip_base = os.path.join(CONFIG["base_dir"], f"quickpull_smartreply_package_{n}")
zip_path = shutil.make_archive(zip_base, "zip", root_dir=CONFIG["deploy_dir"],
                               base_dir=os.path.basename(pkg))
print(f"\nZip: {zip_path} | size {os.path.getsize(zip_path) / 1024**2:.1f} MB")
with zipfile.ZipFile(zip_path) as z:
    for name in z.namelist():
        print("  ", name)
files.download(zip_path)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Module result : ['D21']
Notebook result: ['D21']

Zip: /content/quickpull_smartreply/quickpull_smartreply_package.zip | size 79.7 MB
   smartreply_v1/
   smartreply_v1/__pycache__/
   smartreply_v1/model/
   smartreply_v1/replies_metrics.json
   smartreply_v1/bank_embeddings.npy
   smartreply_v1/dev_messages.csv
   smartreply_v1/golden_test_cases.json
   smartreply_v1/test_messages.csv
   smartreply_v1/requirements.txt
   smartreply_v1/reply_bank.csv
   smartreply_v1/smart_reply.py
   smartreply_v1/replies_config.json
   smartreply_v1/__pycache__/smart_reply.cpython-313.pyc
   smartreply_v1/model/1_Pooling/
   smartreply_v1/model/2_Normalize/
   smartreply_v1/model/tokenizer.json
   smartreply_v1/model/config.json
   smartreply_v1/model/config_sentence_transformers.json
   smartreply_v1/model/model.safetensors
   smartreply_v1/model/sentence_bert_config.json
   smartreply_v1/model/tokenizer_config.json
   smartreply_v1/model/README.md
   smartreply_v1/model/modules.json
   smartreply_v

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>